#  ⭐ `dim_via_administracao`


In [19]:
%run ../../config/bootstrap.py


from pathlib import Path
from utils import get_project_root  

import re
import pandas as pd
from unidecode import unidecode

In [20]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [21]:
path = Path(project_root) / "data/01_bronze/anvisa/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(path)
bronze["VIA_ADMINISTRACAO"] = bronze["VIA_ADMINISTRACAO"].fillna("DESCONHECIDO")

bronze = pd.concat(
    [
        (
            bronze[["VIA_ADMINISTRACAO"]]
            .value_counts()
            .rename("FREQUENCIA_REGISTROS")
            .reset_index()
        ),
        (
            bronze[["VIA_ADMINISTRACAO_MAE_PAI"]].rename(columns={"VIA_ADMINISTRACAO_MAE_PAI": "VIA_ADMINISTRACAO"})
            .value_counts()
            .rename("FREQUENCIA_REGISTROS")
            .reset_index()
        ),
    ],
    ignore_index=True,
)

In [22]:
bronze.head()

,VIA_ADMINISTRACAO,FREQUENCIA_REGISTROS
0,DESCONHECIDO,330705
1,oral,31428
2,infusão intravenosa,26376
3,intramuscular,19896
4,desconhecida,18764


In [23]:
bronze.VIA_ADMINISTRACAO.nunique()

2006

In [24]:
bronze.to_csv(Path(project_root) / "eda/Medicamentos/Medicamentos_VIA_ADM.csv", sep=";", index=False)

# 🥈 Silver

normalized data, medium quality


In [25]:
from utils.med_normalize_via_adm import normalizar_via_fuzzy

In [26]:
silver = pd.read_csv(Path(project_root) / "eda/Medicamentos/Medicamentos_VIA_ADM.csv", sep=";") ## Manual Normalization
 
silver.columns

Index(['VIA_ADMINISTRACAO', 'FREQUENCIA_REGISTROS'], dtype='object')

In [27]:
silver["VIA_STRUCT"] = (
    silver["VIA_ADMINISTRACAO"]
    .astype(str)
    .apply(normalizar_via_fuzzy)
)

silver["VIA_ADMINISTRACAO_CHAVE"] = silver["VIA_STRUCT"].apply(lambda x: x["chave"])
silver["VIA_ADMINISTRACAO_VALOR"] = silver["VIA_STRUCT"].apply(lambda x: x["valor"])
silver["VIA_ADMINISTRACAO_DESCRIPTION"] = silver["VIA_STRUCT"].apply(lambda x: x["description"])
silver["VIA_ADMINISTRACAO_DESCRIPTION_PT"] = silver["VIA_STRUCT"].apply(lambda x: x["description_pt"])

silver.drop(columns="VIA_STRUCT", inplace=True)

In [28]:
silver.columns

Index(['VIA_ADMINISTRACAO', 'FREQUENCIA_REGISTROS', 'VIA_ADMINISTRACAO_CHAVE',
       'VIA_ADMINISTRACAO_VALOR', 'VIA_ADMINISTRACAO_DESCRIPTION',
       'VIA_ADMINISTRACAO_DESCRIPTION_PT'],
      dtype='object')

In [29]:
silver.head()

,VIA_ADMINISTRACAO,FREQUENCIA_REGISTROS,VIA_ADMINISTRACAO_CHAVE,VIA_ADMINISTRACAO_VALOR,VIA_ADMINISTRACAO_DESCRIPTION,VIA_ADMINISTRACAO_DESCRIPTION_PT
0,DESCONHECIDO,330705,0,desconhecido,unknown,Desconhecida
1,oral,31428,5,O,oral,Oral
2,infusão intravenosa,26376,6,P,parenteral,"Parenteral (Injeção intramuscular, Injeção intravenosa, Injeção subcutânea)"
3,intramuscular,19896,6,P,parenteral,"Parenteral (Injeção intramuscular, Injeção intravenosa, Injeção subcutânea)"
4,desconhecida,18764,0,desconhecido,unknown,Desconhecida


In [30]:
(
    silver.groupby("VIA_ADMINISTRACAO_CHAVE")["FREQUENCIA_REGISTROS"]
    .sum()
    .reset_index(name="FREQUENCIA_REGISTROS_TOTAL")
).sort_values(by='FREQUENCIA_REGISTROS_TOTAL', ascending=False)

,VIA_ADMINISTRACAO_CHAVE,FREQUENCIA_REGISTROS_TOTAL
0,0,375275
6,6,154075
5,5,56412
2,2,2076
9,9,1470
3,3,711
1,1,672
4,4,348
8,8,303
10,10,180


In [31]:
silver.query("VIA_ADMINISTRACAO_VALOR == 1").head()

,VIA_ADMINISTRACAO,FREQUENCIA_REGISTROS,VIA_ADMINISTRACAO_CHAVE,VIA_ADMINISTRACAO_VALOR,VIA_ADMINISTRACAO_DESCRIPTION,VIA_ADMINISTRACAO_DESCRIPTION_PT


In [32]:
silver.columns

Index(['VIA_ADMINISTRACAO', 'FREQUENCIA_REGISTROS', 'VIA_ADMINISTRACAO_CHAVE',
       'VIA_ADMINISTRACAO_VALOR', 'VIA_ADMINISTRACAO_DESCRIPTION',
       'VIA_ADMINISTRACAO_DESCRIPTION_PT'],
      dtype='object')

In [33]:
hist = silver.sort_values(by='VIA_ADMINISTRACAO_VALOR').drop(columns="FREQUENCIA_REGISTROS")
hist.head()

,VIA_ADMINISTRACAO,VIA_ADMINISTRACAO_CHAVE,VIA_ADMINISTRACAO_VALOR,VIA_ADMINISTRACAO_DESCRIPTION,VIA_ADMINISTRACAO_DESCRIPTION_PT
1169,INTRATRCAL,1,Implant,implant,Implante
666,Administrado no braço direito,1,Implant,implant,Implante
1887,vai intravenosa,1,Implant,implant,Implante
670,Administrada no braço direito,1,Implant,implant,Implante
671,Administrada no braço esquerdo,1,Implant,implant,Implante


In [34]:
hist.to_parquet(Path(project_root) / "data/02_silver/hist_via_adm/hist_via_adm.parquet", index=False)

# 🥇 Gold


In [35]:
dim = (
    hist[[ "VIA_ADMINISTRACAO_CHAVE","VIA_ADMINISTRACAO_VALOR","VIA_ADMINISTRACAO_DESCRIPTION","VIA_ADMINISTRACAO_DESCRIPTION_PT"]]
    .drop_duplicates()
    .sort_values(by="VIA_ADMINISTRACAO_VALOR")
) 
dim

,VIA_ADMINISTRACAO_CHAVE,VIA_ADMINISTRACAO_VALOR,VIA_ADMINISTRACAO_DESCRIPTION,VIA_ADMINISTRACAO_DESCRIPTION_PT
1169,1,Implant,implant,Implante
1679,2,Inhal,inhalation,Inalatória (pelas vias respiratórias)
1373,3,Instill,instillation,Ocular (nos olhos)
1499,4,N,nasal,Nasal
756,5,O,oral,Oral
1376,6,P,parenteral,"Parenteral (Injeção intramuscular, Injeção intravenosa, Injeção subcutânea)"
386,7,R,rectal,Retal (pelo ânus)
1273,8,SL,sublingual/buccal/oromucosal,Sublingual/Bucal/Oromucosal
1978,9,TD,transdermal,Dérmica (na pele)
548,10,V,vaginal,Vaginal


In [36]:
dim.to_csv(Path(project_root) / "data/03_gold/dim_via_adm/dim_via_adm.csv", sep=";", index=False)